In [19]:
import pandas as pd

In [20]:
df = pd.read_parquet("hf://datasets/sharjeelyunus/github-issues-dataset/github_issues_dataset.parquet")

In [21]:
df.drop(["id", "repo"], axis=1, inplace=True)

## Normalize labels column

In [22]:
import re
import pandas as pd

def clean_labels(raw_labels):
    if pd.isna(raw_labels):
        return ["unknown"]
        
    noise_labels = {"p1", "p2", "p3", "p4", "p5"}
    parts = [l.strip().lower() for l in str(raw_labels).split(",")]
    meaningful = [
        l for l in parts
        if re.search(r"[a-z]", l) and l not in noise_labels
    ]
    return meaningful if meaningful else ["unknown"]

df["labels-list"] = df["labels"].apply(clean_labels)


In [23]:
from collections import Counter

all_labels = [l for row in df["labels-list"] for l in row]
counts = Counter(all_labels)

In [ ]:
import random
import json

with open('map.json', 'r', encoding='utf-8') as f:
    alias_to_group = json.load(f)

def process_labels(labels):
    mapped = []
    seen = set()
    
    for label in labels:
        group = alias_to_group.get(label.lower(), "unknown")
        
        # Consolidate 'other' to 'unknown'
        if group == "other":
            group = "unknown"
            
        if group not in seen:
            mapped.append(group)
            seen.add(group)
    
    # Mapped array resolution
    if len(mapped) > 1 and "unknown" in mapped:
        mapped.remove("unknown")
    final_mapped = mapped if mapped else ["unknown"]
        
    return final_mapped

df["labels-mapped"] = df["labels-list"].apply(process_labels)
df["labels-mapped"].explode().value_counts().head(20)


labels-mapped
bug                     43045
feature_request         31239
ui_ux                   28258
compatibility           17882
build_ci_cd             16969
question                15803
performance             11235
integration_api          9674
testing                  8577
documentation            7201
unknown                  7123
data_import_export       2581
refactor_maintenance     2574
configuration            1766
access_permissions        922
installation_setup        909
security                  732
duplicate_existing         98
Name: count, dtype: int64

In [25]:
target_share = 0.83
total = sum(counts.values())
target = target_share * total

running = 0
n = 0
for _, c in counts.most_common():
    running += c
    n += 1
    if running >= target:
        break

print("n =", n)
print("coverage =", running / total)

n = 528
coverage = 0.8300270540492855


In [26]:
df.sample(10)

,title,body,labels,priority,severity,labels-list,labels-mapped
48029,The codegen unconditionaly generate code even ...,The codegen today generate cpp and header for ...,"triaged,module: codegen",low,Minor,"[triaged, module: codegen]",[build_ci_cd]
49485,FlutterError.reportError is not printed in the...,\r\n\r\n## Steps to Reproduce\r\n\r\nWith the ...,"platform-android,a: error message,has reproduc...",low,Critical,"[platform-android, a: error message, has repro...","[compatibility, ui_ux, bug]"
24995,`error` event not emitted when destroying an H...,* **Version**: v14.5.0\r\n* **Platform**: Linu...,http,low,Critical,[http],[integration_api]
55247,Incorrect debug information generated for test...,Hello!\r\n\r\nWhen investigating an issue with...,"A-debuginfo,T-compiler,C-bug,A-proc-macros",low,Critical,"[a-debuginfo, t-compiler, c-bug, a-proc-macros]","[bug, build_ci_cd]"
24423,RigidBody2D isn't detecting every collision,<!-- Please search existing issues for potenti...,"bug,topic:physics",low,Major,"[bug, topic:physics]",[bug]
42828,[RayWenderlich]: Unable to extract video id,### Checklist\n\n- [X] I'm reporting a broken ...,site-bug,low,Critical,[site-bug],[bug]
77033,Autocorrect highlight does not go away when sc...,### Is there an existing issue for this?\r\n\r...,"a: text input,platform-ios,has reproducible st...",low,Critical,"[a: text input, platform-ios, has reproducible...","[ui_ux, compatibility, bug]"
71310,Increasing batch size makes network forward 10...,### 🐛 Describe the bug\n\nI have a two layer n...,"module: cudnn,module: nn,module: cuda,triaged",low,Critical,"[module: cudnn, module: nn, module: cuda, tria...","[integration_api, performance]"
26861,Go to definition broken by existance of tsconf...,<!-- 🚨 STOP 🚨 STOP 🚨 STOP 🚨\r\n\r\nHalf of all...,"Suggestion,In Discussion",low,Critical,"[suggestion, in discussion]","[feature_request, question]"
76825,[go_router] Responsive ShellRoute,### Is there an existing issue for this?\n\n- ...,"d: examples,package,c: proposal,P3,p: go_route...",low,Critical,"[d: examples, package, c: proposal, p: go_rout...","[documentation, build_ci_cd, feature_request, ..."


In [27]:
# df_clean = df.drop(["labels", "labels-list", "labels-mapped"], axis=1)

In [28]:
# X = df_clean.drop(['priority', 'severity', 'labels-mapped'], axis=1)
# y = df_clean["labels-atomic"]

In [29]:
# X["text"] = X["title"].fillna("") + " [SEP] " + X["body"].fillna("")
# X.drop(["title", "body"], axis=1, inplace=True)

In [30]:
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
# from sklearn.naive_bayes import MultinomialNB
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score, f1_score


# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# models = {
#     "logreg" : LogisticRegression(max_iter=3000, class_weight="balanced"),
#     "svm" : LinearSVC(class_weight="balanced")
# }

# results = []

# for name, clf in models.items():
#     pipe = Pipeline([
#         ("tdif", TfidfVectorizer(
#         lowercase=True,
#         stop_words="english",
#         ngram_range=(1,2),
#         min_df=2, 
#         max_features=50000,
#         sublinear_tf=True)),

#         ("clf", clf)
#     ])

#     pipe.fit(X_train["text"], y_train)
#     y_pred = pipe.predict(X_test["text"])

#     acc_score = accuracy_score(y_test, y_pred)
#     f1_val = f1_score(y_test, y_pred, average='macro')

#     results.append((name, acc_score, f1_val))
#     print(f"\n{name}")
#     print("acc_scoreuracy:", acc_score)
#     print("macro f1_val:", f1_val)
#     from sklearn.metrics import classification_report
#     print(classification_report(y_test, y_pred))
